In [1]:
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableSequence
from langchain_ollama import ChatOllama
from collections.abc import Sequence, Mapping

In [2]:
# Install ollama.
# Download gemma4:e4b using "ollama pull gemma4:e4b"
class LocalLLM:
    def __init__(
        self,
        model: str='gemma4:e4b',
        temperature: float=0.7,
    ):
        self._model = model
        self._temperature = temperature
        
    def __call__(self):
        return ChatOllama(
            model=self._model,
            temperature=self._temperature,
            validate_model_on_init=True,
        )

In [3]:
class BookSummarizer:
    """Summarizes a book, providing both a narrative summary, and a bulleted list."""

    def __init__(
        self,
        llm: LocalLLM,
        chunk_size: int = 2500,
        chunk_overlap: int = 100,
    ) -> None:
        self._llm = llm()
        self._text_splitter = TokenTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        self._cached_chain = self._get_chain()

    def _split_into_chunks(self, book_input: str) -> Sequence[Mapping[str, str]]:
        """Chunks text and returns a Sequence of Mappings."""
        return [
            {'chunk': chunk}
            for chunk in self._text_splitter.split_text(book_input)
        ]

    def _map_chain(self) -> RunnableSequence:
        """Summarize each chunk."""
        map_prompt_template = '''
        Write a concise summary of the following text, and include the main details.
        Text: {chunk}
        '''
        map_prompt = PromptTemplate.from_template(map_prompt_template)
        return map_prompt | self._llm | StrOutputParser()
        
    def _combine_summaries(self, summaries: Sequence[str]) -> Mapping[str, str]:
        """Combine summaries for each chunk."""
        return {'summaries': '\n'.join(summaries)}

    def _reduce_chain(self) -> RunnableSequence:
        """Provides a narrative summary of individual summaries."""
        reduce_prompt_template = '''
        Write a concise summary of the following text, which joins several summaries, and include the main details.
        Text: {summaries}
        '''
        reduce_prompt = PromptTemplate.from_template(reduce_prompt_template)
        return reduce_prompt | self._llm | StrOutputParser()

    def _bullet_chain(self) -> RunnableSequence:
        """Provides a bullet point summary of individual summaries."""
        bullet_prompt_template = '''
        List 5 key takeaways from these summaries.
        Text: {summaries}
        '''
        bullet_prompt = PromptTemplate.from_template(bullet_prompt_template)
        return bullet_prompt | self._llm | StrOutputParser()

    def _get_chain(self) -> RunnableSequence:
        """Assembles and returns the full LCEL chain."""
        return (
            RunnableLambda(self._split_into_chunks)
            | self._map_chain().map() 
            | RunnableLambda(self._combine_summaries)
            | RunnableParallel({
                'narrative_summary': self._reduce_chain(),
                'bullet_points': self._bullet_chain(),
            })
        )

    def summarize(self, book: str) -> Mapping[str, str]:
        """Summarize a book and return a narrative summary, as well as bullet points.

        Args:
            book: Input book to be summarized.
        Returns:
            The summary including:
                'narrative_summary': The narrative summary of the book.
                'bullet_points': Bullet point summary.
        """
        if not book:
            return {'narrative_summary': '', 'bullet_points': ''}
        return self._cached_chain.invoke(book)        


In [4]:
llm = LocalLLM()

In [5]:
# Read some input:
with open('docs/moby_dick_chapters_1-3.txt', 'r', encoding='utf-8') as fptr:
    book = fptr.read()

# Summarize.
summarizer = BookSummarizer(llm)
summary = summarizer.summarize(book)

In [6]:
print(summary['narrative_summary'])
print()
print('*-'*50)
print()
print(summary['bullet_points'])

This text provides a multi-layered account of the narrator's journey to sea, blending philosophical musing, travel narrative, and dramatic encounter.

**Philosophical and Preparatory Stages:**
The narrator, Ishmael, establishes that his attraction to the sea is profound, viewing water as a powerful, magnetic force representing the "ungraspable phantom of life." He clarifies that his decision to become a sailor is not for profit but is driven by a deep, almost mystical connection to the ocean, combined with the need for physical exercise and fresh air. His journey is framed by philosophical reflections on fate and destiny, suggesting his whaling voyage is part of a predetermined "grand programme of Providence."

**The Journey and Lodging:**
Traveling from Manhattan, the narrator heads to New Bedford, struggling with delays that prevent him from reaching his desired whaling center, Nantucket. Upon arriving at the cold, dismal "Spouter Inn," he is immersed in the unique atmosphere of the 